# Lecture 1 · Linear Models for Classification I — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise the material of **Lecture 1**: the geometry of a linear discriminant $y(\mathbf{x}) = \mathbf{w}^T\mathbf{x} + w_0$
(normal vector, signed distance, decision regions), the multi-class strategies (One-Against-All, One-Against-One and
joint discriminants $y_k(\mathbf{x})$ with `argmax`), and **least-squares classification** with one-hot targets,
including the closed form $\tilde{\mathbf{W}} = (\tilde{\mathbf{X}}^T\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^T\mathbf{T}$
and why it can fail. The by-hand exercises reuse the numbers from the lecture notebook (`lecture1.ipynb`) and
[[Examples]]; the code exercises rebuild those experiments from scratch in NumPy.

**18 exercises** · 🧠 Concept ×4 · ✍️ By hand ×8 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import LinearRegression, LogisticRegression, Perceptron
from sklearn.svm import SVC
from scipy.stats import multivariate_normal as mvn
rng = np.random.default_rng(1)

# The three linear functions from the lecture notebook (Experiment 3)
def y1(x1, x2): return 2 - 2/3 * x1 - x2
def y2(x1, x2): return 3 + 3 * x1 - x2
def y3(x1, x2): return 1 + 0.1 * x1 - 0.25 * x2

# Experiment 2 data, columns = samples (no bias row, exactly as in the lecture)
X_cols = np.array([[1., 1., 0.],
                   [1., 5., 1.]])
T_cols = np.array([[1., 1., 0.],
                   [0., 0., 1.]])

## Part A · Concepts

### Exercise 1 · What do w and w₀ control?
*🧠 Concept · ★☆☆*

For $y(\mathbf{x}) = \mathbf{w}^T\mathbf{x} + w_0$ in $\mathbb{R}^2$, say what happens to the **decision boundary** and to the
**decision regions** $\mathcal{R}_1, \mathcal{R}_2$ when you

1. multiply both $\mathbf{w}$ and $w_0$ by $c = 5$,
2. multiply both by $c = -1$,
3. change only $w_0$,
4. rotate only $\mathbf{w}$.

Which of these change the value $y(\mathbf{x})/\lVert\mathbf{w}\rVert$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The boundary is the set $\{\mathbf{x}: y(\mathbf{x}) = 0\}$. Which operations keep that set, and which flip the sign of $y$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $c=5$: same zero set, so **same boundary and same regions**; $y(\mathbf{x})$ grows by 5 but $y/\lVert\mathbf{w}\rVert$ (the signed distance) is unchanged.
2. $c=-1$: same boundary line, but $\mathcal{R}_1$ and $\mathcal{R}_2$ **swap** (the normal now points the other way); the signed distance flips sign.
3. Changing $w_0$ **translates** the boundary parallel to itself (its distance to the origin is $-w_0/\lVert\mathbf{w}\rVert$ along $\mathbf{w}/\lVert\mathbf{w}\rVert$).
4. Rotating $\mathbf{w}$ rotates the boundary: $\mathbf{w}$ is always its normal.

Only positive rescaling leaves both the classifier and the signed distance untouched. This scale invariance is exactly what the SVM
exploits later when it fixes $y_n(\mathbf{w}^T\mathbf{x}_n+b)=1$ for the closest points ([[Lecture Notes 1-12#Lecture 2: Linear Models for Classification II]]).

</details>

### Exercise 2 · Counting classifiers and ambiguity
*🧠 Concept · ★☆☆*

(a) How many binary classifiers do One-Against-All and One-Against-One need for $K = 4$ and for $K = 10$ classes?

(b) Describe the two kinds of *ambiguous* region One-Against-All can produce, and the one One-Against-One produces.

(c) Why can the joint discriminant rule "$\mathbf{x}\in\mathcal{C}_k$ iff $y_k(\mathbf{x}) > y_j(\mathbf{x})\ \forall j\neq k$" never leave a point unassigned?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

OvO trains one classifier per unordered pair: $\binom{K}{2}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) OvA: $K$ classifiers, so 4 and 10. OvO: $K(K-1)/2$, so **6** and **45**.

(b) OvA: a point can be claimed by **several** classifiers ("I am class $k$" from more than one) or by **none** of them.
OvO: majority voting can **deadlock**, e.g. a cyclic $a\succ b$, $b\succ c$, $c\succ a$ gives every class one vote.

(c) For any $\mathbf{x}$ the finite set $\{y_1(\mathbf{x}),\dots,y_K(\mathbf{x})\}$ has a maximum, so `argmax` always returns a class; only the
measure-zero set where two scores tie exactly is a boundary. That is why the joint discriminant has no unassigned territory.

</details>

### Exercise 3 · Why least squares is a bad classification loss
*🧠 Concept · ★★☆*

The lecture argues that the sum-of-squares error $E_D(\tilde{\mathbf{W}})$ is a poor objective for classification.

(a) From which probabilistic assumption does the sum-of-squares error come, and why is it a strange assumption for one-hot targets?

(b) A point of class 1 gets the score vector $\mathbf{y} = (4, -3)^T$ with target $\mathbf{t} = (1, 0)^T$. Is it classified correctly? What is its
contribution $\frac12\lVert\mathbf{y}-\mathbf{t}\rVert^2$, and what does minimising this term do to the decision boundary?

(c) Give one loss used later in the course that does **not** have this problem.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Least squares = maximum likelihood with Gaussian noise around the target. A Gaussian is symmetric: being 'too right' costs as much as being wrong.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Minimising SSE is maximum likelihood under $\mathbf{t} = \tilde{\mathbf{W}}^T\tilde{\mathbf{x}} + \boldsymbol\varepsilon$ with Gaussian $\boldsymbol\varepsilon$.
One-hot targets are binary, so a symmetric unbounded noise model around them makes no sense.

(b) Yes, $\arg\max = 1$ is correct, but the loss is $\frac12[(4-1)^2 + (-3-0)^2] = 9$, larger than many *misclassified* points would
produce. To reduce it the optimiser pulls the scores of this "over-correct" point back towards the targets, which tilts and shifts the
boundary towards it: far-away correct points act like **outliers** that drag the boundary.

(c) The cross-entropy of logistic regression (bounded, nearly linear penalty for confident correct points) or the hinge loss of the SVM
(zero penalty once $y_n f(\mathbf{x}_n) \ge 1$).

</details>

### Exercise 4 · Same capacity, different boundaries
*🧠 Concept · ★★☆*

The Perceptron, Naive Bayes (binary features) and least-squares classification all produce **linear** decision boundaries.
(a) What does it mean that they have the *same capacity*? (b) Why do they nevertheless end up with different boundaries on the same data?
(c) Which one would you expect to change the most if you add 20 correctly-classified points very far from the boundary, and why?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Capacity = the set of functions the model *can* represent. The learned function depends on the objective optimised.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Each can represent exactly the set of half-space classifiers $\operatorname{sign}(\mathbf{w}^T\mathbf{x}+w_0)$; none can represent XOR.

(b) They optimise different criteria: the Perceptron corrects misclassified points only, Naive Bayes maximises a joint likelihood with a
feature-independence assumption, least squares minimises squared distance of scores to 0/1 targets. Different objectives select different
members of the same hypothesis set, with different sensitivity to noise, outliers and class densities.

(c) **Least squares**: far correct points have large squared residuals and pull the boundary (Exercise 16 shows it). The Perceptron ignores
points that are already correct, so it does not move at all.

</details>

## Part B · By hand

### Exercise 5 · Distance to the origin
*✍️ By hand · ★☆☆*

Take $w_0 = 2.5$ and $\mathbf{w} = (1.3, 1.3)^T$ (Example 4 in [[Examples]]).

(a) Write the boundary as $x_2 = a x_1 + c$.
(b) Compute the distance from the origin to the boundary, $|w_0|/\lVert\mathbf{w}\rVert$.
(c) Compute the signed distance $y(\mathbf{x})/\lVert\mathbf{w}\rVert$ of the point $\mathbf{x} = (1, 1)^T$. On which side is it?

In [ ]:
dist_origin = None   # (b)
signed_dist_11 = None  # (c)

_w = np.array([1.3, 1.3]); _w0 = 2.5
check('distance origin', dist_origin, abs(_w0) / np.linalg.norm(_w), tol=1e-2)
check('signed distance of (1,1)', signed_dist_11, (_w @ np.ones(2) + _w0) / np.linalg.norm(_w), tol=1e-2)

<details>
<summary><b>💡 Hint</b></summary>

$\lVert\mathbf{w}\rVert = \sqrt{1.3^2+1.3^2} = \sqrt{3.38}\approx 1.8385$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $2.5 + 1.3x_1 + 1.3x_2 = 0 \Rightarrow x_2 = -x_1 - 2.5/1.3 \approx -x_1 - 1.923$.

(b) $2.5/\sqrt{3.38} \approx 2.5/1.8385 \approx \mathbf{1.360}$.

(c) $y(1,1) = 2.5 + 1.3 + 1.3 = 5.1$, so $5.1/1.8385 \approx \mathbf{2.774}$; positive, so $(1,1)$ is in $\mathcal{R}_1$ (the side $\mathbf{w}$ points to).

```python
dist_origin = 2.5 / np.sqrt(3.38)
signed_dist_11 = 5.1 / np.sqrt(3.38)
```

</details>

### Exercise 6 · Orthogonality of w (Experiment 1)
*✍️ By hand · ★☆☆*

For $y_1(x_1,x_2) = 2 - \tfrac23 x_1 - x_2$:

(a) Read off $\mathbf{w}$ and $w_0$. (b) Take the two boundary points $A = (0, 2)$, $B = (3, 0)$, form $\mathbf{x}_\perp = A - B$ and compute $\mathbf{w}^T\mathbf{x}_\perp$.
(c) Which side of the line contains the origin? (d) Compute the distance from the origin to the line.

In [ ]:
w_dot_xperp = None  # (b)
dist_line = None    # (d)

check('w^T x_perp', w_dot_xperp, np.array([-2/3, -1]) @ (np.array([0, 2.]) - np.array([3, 0.])))
check('distance origin', dist_line, 2 / np.linalg.norm([-2/3, -1]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\mathbf{w} = (-\tfrac23, -1)^T$, $w_0 = 2$. The distance is $|w_0|/\lVert\mathbf{w}\rVert$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\mathbf{w} = (-\tfrac23, -1)^T$, $w_0 = 2$.
(b) $\mathbf{x}_\perp = (-3, 2)^T$, $\mathbf{w}^T\mathbf{x}_\perp = 2 - 2 = \mathbf{0}$: $\mathbf{w}$ is normal to the line.
(c) $y_1(0,0) = 2 > 0$, so the origin is in $\mathcal{R}_1$: the half-plane **below/left** of the line. Note $\mathbf{w}$ points down-left, towards the origin.
(d) $\lVert\mathbf{w}\rVert = \sqrt{4/9 + 1} = \sqrt{13}/3 \approx 1.2019$, distance $= 2/1.2019 \approx \mathbf{1.664}$.

```python
w_dot_xperp = 0.0
dist_line = 2 / (np.sqrt(13) / 3)
```

</details>

### Exercise 7 · Deriving the signed distance
*✍️ By hand · ★★☆*

Write any point as $\mathbf{x} = \mathbf{x}_\perp + r\,\frac{\mathbf{w}}{\lVert\mathbf{w}\rVert}$ where $\mathbf{x}_\perp$ is its orthogonal projection onto
the hyperplane $y(\mathbf{x}) = 0$. Show that $r = y(\mathbf{x})/\lVert\mathbf{w}\rVert$, and deduce the distance from the origin to the hyperplane.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Apply $\mathbf{w}^T(\cdot) + w_0$ to both sides and use $y(\mathbf{x}_\perp) = 0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$y(\mathbf{x}) = \mathbf{w}^T\mathbf{x}_\perp + w_0 + r\,\frac{\mathbf{w}^T\mathbf{w}}{\lVert\mathbf{w}\rVert} = y(\mathbf{x}_\perp) + r\lVert\mathbf{w}\rVert = r\lVert\mathbf{w}\rVert$,
hence $r = \dfrac{y(\mathbf{x})}{\lVert\mathbf{w}\rVert}$. For $\mathbf{x} = \mathbf{0}$: $r = w_0/\lVert\mathbf{w}\rVert$. Its absolute value is the distance;
its sign tells on which side of the plane the origin lies.

</details>

### Exercise 8 · Joint discriminants by hand
*✍️ By hand · ★★☆*

Use the three functions from Experiment 3 as joint discriminants (scores $y_a = y_1$, $y_b = y_2$, $y_c = y_3$):
$$y_1 = 2 - \tfrac23 x_1 - x_2,\quad y_2 = 3 + 3x_1 - x_2,\quad y_3 = 1 + \tfrac1{10}x_1 - \tfrac14 x_2 .$$

(a) Classify the points $P = (0,0)$, $Q = (-2, 0)$, $R = (0, 4)$ (answer with class numbers 1, 2, 3).
(b) Write the boundary between classes 1 and 2 as an equation $(\mathbf{w}_1-\mathbf{w}_2)^T\mathbf{x} + (w_{10}-w_{20}) = 0$ and simplify it.

In [ ]:
labels_PQR = None   # e.g. [1, 2, 3]
boundary_12_x1 = None  # the boundary 1|2 is the vertical line x1 = ?

_P = np.array([[0, 0], [-2, 0], [0, 4.]])
_S = np.stack([y1(*_P.T), y2(*_P.T), y3(*_P.T)], 1)
check('labels', labels_PQR, np.argmax(_S, 1) + 1)
from scipy.optimize import brentq
check('boundary 1|2', boundary_12_x1, brentq(lambda a: y1(a, 0.) - y2(a, 0.), -5, 5))

<details>
<summary><b>💡 Hint 1</b></summary>

Just evaluate all three scores at each point and take the largest.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$y_1 - y_2 = -1 - \tfrac{11}{3}x_1$; the $x_2$ terms cancel.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $P$: $(2, 3, 1)\Rightarrow$ class **2**. $Q$: $(3.33, -3, 0.8)\Rightarrow$ class **1**. $R$: $(-2, -1, 0)\Rightarrow$ class **3**.

(b) $y_1 - y_2 = (2-3) + (-\tfrac23 - 3)x_1 + (-1+1)x_2 = -1 - \tfrac{11}{3}x_1 = 0 \Rightarrow x_1 = -\tfrac{3}{11} \approx -0.273$.
The 1|2 boundary is a vertical line because $\mathbf{w}_1-\mathbf{w}_2$ has no $x_2$ component. (Only the part of this line where $y_3$ is
not the largest is an actual boundary.)

```python
labels_PQR = [2, 1, 3]
boundary_12_x1 = -3 / 11
```

</details>

### Exercise 9 · Least squares picks the wrong model
*✍️ By hand · ★★☆*

Data (columns are samples): $\mathbf{x}_1 = (1,1)^T, \mathbf{x}_2 = (1,5)^T$ in class 1 and $\mathbf{x}_3 = (0,1)^T$ in class 2,
targets $\mathbf{t}_1 = \mathbf{t}_2 = (1,0)^T$, $\mathbf{t}_3 = (0,1)^T$. Scores are $\mathbf{y} = W\mathbf{x}$ with (rows = classes)
$$W_1 = \begin{pmatrix}10 & 0\\ -1 & 1\end{pmatrix},\qquad W_2 = \begin{pmatrix}1 & 1\\ 1 & -1\end{pmatrix}.$$

For each model compute the predictions $W\mathbf{x}_n$, the accuracy, and $E_D = \tfrac12\sum_n\lVert W\mathbf{x}_n-\mathbf{t}_n\rVert^2$.
Which model would least squares prefer?

In [ ]:
E_D_W1 = None
E_D_W2 = None
acc_W1 = None
acc_W2 = None

def _E(W): return 0.5 * np.trace((W @ X_cols - T_cols).T @ (W @ X_cols - T_cols))
def _acc(W): return np.mean(np.argmax(W @ X_cols, 0) == np.argmax(T_cols, 0))
_W1 = np.array([[10., 0], [-1, 1]]); _W2 = np.array([[1., 1], [1, -1]])
check('E_D(W1)', E_D_W1, _E(_W1)); check('E_D(W2)', E_D_W2, _E(_W2))
check('accuracy W1', acc_W1, _acc(_W1), tol=1e-2); check('accuracy W2', acc_W2, _acc(_W2), tol=1e-2)

<details>
<summary><b>💡 Hint 1</b></summary>

$W_1$: $\mathbf{y}_1 = (10, 0)$, $\mathbf{y}_2 = (10, 4)$, $\mathbf{y}_3 = (0, 1)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$W_2$: $\mathbf{y}_1 = (2, 0)$, $\mathbf{y}_2 = (6,-4)$, $\mathbf{y}_3 = (1,-1)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$W_1$: residuals $(9,0), (9,4), (0,0)$, so $E_D = \tfrac12(81 + 81 + 16) = \mathbf{89}$, accuracy $3/3 = 1$.

$W_2$: residuals $(1,0), (5,-4), (1,-2)$, so $E_D = \tfrac12(1 + 41 + 5) = \mathbf{23.5}$, accuracy $2/3$ ($\mathbf{x}_3$ gets $(1,-1)$, i.e. class 1).

Least squares prefers $W_2$ ($23.5 < 89$) although it misclassifies a point: SSE is not aligned with accuracy, because $\mathbf{x}_2$ is
"too correct" under $W_1$ and costs a lot.

```python
E_D_W1 = 89.0
E_D_W2 = 23.5
acc_W1 = 1.0
acc_W2 = 2/3
```

</details>

### Exercise 10 · The optimal W₃ by hand
*✍️ By hand · ★★★*

Same data as the previous exercise. Using the columns-as-samples closed form
$$W = T X^T (X X^T)^{-1},$$
compute $X X^T$, its inverse, $T X^T$ and finally $W_3$ by hand (fractions!). Then compute $E_D(W_3)$ and check that $W_3$ classifies all
three points correctly.

In [ ]:
W3_hand = None   # 2x2 array, rows = classes
E_D_W3 = None

_W = T_cols @ X_cols.T @ np.linalg.inv(X_cols @ X_cols.T)
check('W3', W3_hand, _W, tol=1e-3)
check('E_D(W3)', E_D_W3, 0.5 * np.sum((_W @ X_cols - T_cols) ** 2), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$XX^T = \begin{pmatrix}2 & 6\\ 6 & 27\end{pmatrix}$, determinant $54 - 36 = 18$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$TX^T = \begin{pmatrix}2 & 6\\ 0 & 1\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(XX^T)^{-1} = \tfrac1{18}\begin{pmatrix}27 & -6\\ -6 & 2\end{pmatrix}$, so
$$W_3 = \begin{pmatrix}2&6\\0&1\end{pmatrix}\tfrac1{18}\begin{pmatrix}27&-6\\-6&2\end{pmatrix}
= \tfrac1{18}\begin{pmatrix}18 & 0\\ -6 & 2\end{pmatrix} = \begin{pmatrix}1 & 0\\ -\tfrac13 & \tfrac19\end{pmatrix}.$$
Scores: $\mathbf{x}_1 \mapsto (1, -\tfrac29)$, $\mathbf{x}_2 \mapsto (1, \tfrac29)$, $\mathbf{x}_3 \mapsto (0, \tfrac19)$: all three correct.
Residuals in the second row: $-\tfrac29, \tfrac29, -\tfrac89$, first row all zero, so
$E_D = \tfrac12\left(\tfrac4{81}+\tfrac4{81}+\tfrac{64}{81}\right) = \tfrac{36}{81} = \tfrac49 \approx \mathbf{0.444}$.
Here the global optimum happens to classify correctly; the previous exercise shows that is not guaranteed.

```python
W3_hand = np.array([[1, 0], [-1/3, 1/9]])
E_D_W3 = 4 / 9
```

</details>

### Exercise 11 · Normal equations for the multi-target case
*✍️ By hand · ★★☆*

Starting from $E_D(\tilde{\mathbf{W}}) = \tfrac12\operatorname{Tr}\{(\tilde{\mathbf{X}}\tilde{\mathbf{W}}-\mathbf{T})^T(\tilde{\mathbf{X}}\tilde{\mathbf{W}}-\mathbf{T})\}$
(rows = samples), derive $\tilde{\mathbf{W}} = (\tilde{\mathbf{X}}^T\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^T\mathbf{T}$. Then show that, with a bias
column, the $K$ predicted scores of **any** input always sum to 1. (Why does that not make them probabilities?)

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint 1</b></summary>

The trace objective splits into $K$ independent least-squares problems, one per column $\mathbf{t}_{:,k}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For the sum: the vector of ones $\mathbf{1}_K$ satisfies $\mathbf{T}\mathbf{1}_K = \mathbf{1}_N$, which is the bias column of $\tilde{\mathbf{X}}$, so it is fitted exactly.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$E_D = \tfrac12\sum_k \lVert\tilde{\mathbf{X}}\tilde{\mathbf{w}}_k - \mathbf{t}_{:,k}\rVert^2$; each term gives
$\tilde{\mathbf{X}}^T(\tilde{\mathbf{X}}\tilde{\mathbf{w}}_k - \mathbf{t}_{:,k}) = 0$. Stacking columns:
$\tilde{\mathbf{X}}^T\tilde{\mathbf{X}}\tilde{\mathbf{W}} = \tilde{\mathbf{X}}^T\mathbf{T}$, so $\tilde{\mathbf{W}} = (\tilde{\mathbf{X}}^T\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^T\mathbf{T}$.

Sum of scores: $\tilde{\mathbf{W}}\mathbf{1}_K = (\tilde{\mathbf{X}}^T\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^T\mathbf{T}\mathbf{1}_K = (\tilde{\mathbf{X}}^T\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^T\mathbf{1}_N = \mathbf{e}_1$
(regressing the all-ones vector on a design that contains it returns weight 1 on the bias and 0 elsewhere). Thus
$\mathbf{1}_K^T\mathbf{y}(\mathbf{x}) = \mathbf{e}_1^T\tilde{\mathbf{x}} = 1$. The scores are **not** constrained to $[0,1]$ (they can be negative or
above 1), so they are not probabilities. Exercise 14 checks this numerically.

</details>

### Exercise 12 · Joint discriminant regions are convex
*✍️ By hand · ★★☆*

Prove that each decision region $\mathcal{R}_k = \{\mathbf{x} : y_k(\mathbf{x}) > y_j(\mathbf{x})\ \forall j\neq k\}$ of a joint linear discriminant is convex:
if $\mathbf{x}_A, \mathbf{x}_B\in\mathcal{R}_k$ then $\hat{\mathbf{x}} = \lambda\mathbf{x}_A + (1-\lambda)\mathbf{x}_B\in\mathcal{R}_k$ for all $\lambda\in[0,1]$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Linear (affine) functions satisfy $y_k(\hat{\mathbf{x}}) = \lambda y_k(\mathbf{x}_A) + (1-\lambda)y_k(\mathbf{x}_B)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For every $j\neq k$: $y_k(\hat{\mathbf{x}}) = \lambda y_k(\mathbf{x}_A) + (1-\lambda)y_k(\mathbf{x}_B) > \lambda y_j(\mathbf{x}_A) + (1-\lambda)y_j(\mathbf{x}_B) = y_j(\hat{\mathbf{x}})$,
using both inequalities $y_k > y_j$ at $\mathbf{x}_A$ and $\mathbf{x}_B$ with non-negative weights (at least one positive). Hence $\hat{\mathbf{x}}\in\mathcal{R}_k$.
Equivalently, $\mathcal{R}_k$ is an intersection of the open half-spaces $(\mathbf{w}_k-\mathbf{w}_j)^T\mathbf{x} + (w_{k0}-w_{j0}) > 0$, and intersections of convex sets are convex.
Consequence: every region is singly connected, with no holes and no unassigned pockets.

</details>

## Part C · Code from scratch

### Exercise 13 · Discriminant, side and signed distance
*💻 Code · ★☆☆*

Implement `signed_distance(X, w, w0)` returning $y(\mathbf{x}_n)/\lVert\mathbf{w}\rVert$ for every row of `X`, and `side(X, w, w0)` returning `1`
for $\mathcal{R}_1$ ($y\ge 0$) and `2` for $\mathcal{R}_2$. Test on Example 4 ($\mathbf{w} = (1.3, 1.3)$, $w_0 = 2.5$).

In [ ]:
def signed_distance(X, w, w0):
    # TODO
    return None

def side(X, w, w0):
    # TODO: 1 if y(x) >= 0 else 2
    return None

X_test = np.array([[0., 0.], [1., 1.], [-3., -1.], [-1., -0.923]])
d_test = signed_distance(X_test, np.array([1.3, 1.3]), 2.5)
s_test = side(X_test, np.array([1.3, 1.3]), 2.5)

_w = np.array([1.3, 1.3])
# reference: distance via orthogonal projection onto the line, with sign from the normal
_p0 = np.array([0., -2.5 / 1.3])                   # a point on the line
_ref = (X_test - _p0) @ (_w / np.linalg.norm(_w))
check('signed distances', d_test, _ref)
check('sides', s_test, np.where(_ref >= 0, 1, 2))

<details>
<summary><b>💡 Hint</b></summary>

Vectorise: `(X @ w + w0) / np.linalg.norm(w)`; `np.where(cond, 1, 2)` for the side.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The point $(-1,-0.923)$ lies almost on the line, so its distance is $\approx 0$; the check computes distances in an independent way (projection onto the unit normal from a point on the line).

```python
def signed_distance(X, w, w0):
    return (X @ w + w0) / np.linalg.norm(w)

def side(X, w, w0):
    return np.where(X @ w + w0 >= 0, 1, 2)

X_test = np.array([[0., 0.], [1., 1.], [-3., -1.], [-1., -0.923]])
d_test = signed_distance(X_test, np.array([1.3, 1.3]), 2.5)
s_test = side(X_test, np.array([1.3, 1.3]), 2.5)
```

</details>

### Exercise 14 · Least-squares classifier on Iris
*💻 Code · ★★☆*

Implement `fit_ls(X, y, K)` that builds the augmented design $\tilde{\mathbf{X}}$ (bias column first) and one-hot targets $\mathbf{T}$, and returns
$\tilde{\mathbf{W}}\in\mathbb{R}^{(D+1)\times K}$; and `predict_ls(W, X)` = `argmax` of the scores. Use `np.linalg.solve` or `lstsq`.
Fit it on Iris with features 1 and 2 (as in Self Study 1) and store the training accuracy in `acc_ls`.
Also store `score_sums`: the row sums of the score matrix on the training data (see Exercise 11).

In [ ]:
iris = load_iris()
X_ir, y_ir = iris.data[:, [1, 2]], iris.target

def fit_ls(X, y, K):
    # TODO: Xt = [1, X], T = one-hot(y), solve normal equations
    return None

def predict_ls(W, X):
    # TODO
    return None

W_ir = fit_ls(X_ir, y_ir, 3)
acc_ls = None
score_sums = None

_T = np.eye(3)[y_ir]
_lr = LinearRegression().fit(X_ir, _T)          # sklearn's multi-output least squares
_ref_pred = np.argmax(_lr.predict(X_ir), 1)
check('LS accuracy (same as sklearn LinearRegression on one-hot)', acc_ls, np.mean(_ref_pred == y_ir))
check('scores sum to 1', score_sums, np.ones(len(y_ir)))

<details>
<summary><b>💡 Hint</b></summary>

`T = np.eye(K)[y]` gives the one-hot matrix in one line.

</details>

<details>
<summary><b>✅ Solution</b></summary>

This is exactly what `LinearRegression().fit(X, onehot)` does in Self Study 1. The training accuracy is only about **0.80**
(logistic regression gets about 0.95 on the same two features): the middle class (versicolor) is partly **masked**, because its score is never the
largest over a wide enough strip. This is another least-squares failure mode. The score sums are exactly 1
(Exercise 11) even though individual scores leave $[0,1]$.

```python
iris = load_iris()
X_ir, y_ir = iris.data[:, [1, 2]], iris.target

def fit_ls(X, y, K):
    Xt = np.c_[np.ones(len(X)), X]
    T = np.eye(K)[y]
    return np.linalg.solve(Xt.T @ Xt, Xt.T @ T)

def predict_ls(W, X):
    return np.argmax(np.c_[np.ones(len(X)), X] @ W, axis=1)

W_ir = fit_ls(X_ir, y_ir, 3)
acc_ls = np.mean(predict_ls(W_ir, X_ir) == y_ir)
score_sums = (np.c_[np.ones(len(X_ir)), X_ir] @ W_ir).sum(1)
print('accuracy', acc_ls, ' min/max score', (np.c_[np.ones(len(X_ir)), X_ir] @ W_ir).min().round(2), (np.c_[np.ones(len(X_ir)), X_ir] @ W_ir).max().round(2))
```

</details>

### Exercise 15 · One-Against-All ambiguity vs. argmax
*💻 Code · ★★☆*

Use $y_1, y_2, y_3$ (from the setup) as **One-Against-All** classifiers: classifier $k$ claims $\mathbf{x}$ if $y_k(\mathbf{x}) > 0$.
On the grid $x_1, x_2\in[-5, 5]$ (201 × 201 points, `np.linspace`), compute

* `frac_ambiguous_ova`: the fraction of grid points claimed by **zero or more than one** classifier,
* `frac_unassigned_joint`: the fraction left unassigned by the joint `argmax` rule (it should be 0).

Optional: visualise both with `plt.contourf`.

In [ ]:
g = np.linspace(-5, 5, 201)
G1, G2 = np.meshgrid(g, g)
frac_ambiguous_ova = None
frac_unassigned_joint = None

_n_claims = sum(((f(G1, G2) > 0).astype(int) for f in (y1, y2, y3)))
check('OvA ambiguous fraction', frac_ambiguous_ova, np.mean(_n_claims != 1))
check('joint argmax leaves nothing unassigned', frac_unassigned_joint, 0.0)

<details>
<summary><b>💡 Hint</b></summary>

Stack the three score grids into an array of shape (3, 201, 201); count positives along axis 0.

</details>

<details>
<summary><b>✅ Solution</b></summary>

About 92% of the square is ambiguous for OvA (claimed by none or by several classifiers), while the joint rule assigns every point to exactly one class.
The OvA lines and the joint boundaries are **different** lines: joint boundaries are where scores tie ($y_k = y_j$), not where a score crosses zero.

```python
g = np.linspace(-5, 5, 201)
G1, G2 = np.meshgrid(g, g)
S = np.stack([y1(G1, G2), y2(G1, G2), y3(G1, G2)])
claims = (S > 0).sum(0)
frac_ambiguous_ova = np.mean(claims != 1)
labels_joint = np.argmax(S, 0)                       # always defined
frac_unassigned_joint = 0.0 if labels_joint.shape == G1.shape else 1.0
print('OvA ambiguous:', round(frac_ambiguous_ova, 3))
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].contourf(G1, G2, np.where(claims == 1, np.argmax(S > 0, 0), -1), levels=[-1.5, -0.5, 0.5, 1.5, 2.5]); ax[0].set_title('One-Against-All (dark = ambiguous)')
ax[1].contourf(G1, G2, labels_joint, levels=[-0.5, 0.5, 1.5, 2.5]); ax[1].set_title('joint argmax')
plt.show()
```

</details>

### Exercise 16 · Outliers drag the least-squares boundary
*💻 Code · ★★★*

Create two Gaussian classes (50 points each) around $(-1, 1)$ and $(1,-1)$ with std 0.6 (`rng` from the setup). Fit your `fit_ls`
and record the accuracy on these 100 points (`acc_before`). Then add 20 **extra class-1 points** around $(7, -6)$ (std 0.5): they are far on the
*correct* side. Refit least squares on all 120 points and record the accuracy **on the original 100 points** (`acc_after`).
Do the same with `LogisticRegression()` (`acc_logreg_after`). Explain.

In [ ]:
acc_before = None
acc_after = None
acc_logreg_after = None

if acc_before is not None:
    check('LS gets worse after adding correct points', acc_after < acc_before, True)
    check('logistic regression is not fooled (>= 0.97)', acc_logreg_after >= 0.97, True)
else:
    check('LS gets worse after adding correct points', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Keep the original `X, y` separate from the augmented `Xa, ya`, and always evaluate on `X, y`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Least squares drops from about 0.99 to about 0.93: the 20 far points have large residuals ("too correct"), so the fit rotates the boundary towards them,
and points near the original boundary end up on the wrong side. Logistic regression's loss is almost zero for confidently-correct points, so the far
cluster barely matters. This is the outlier and over-confidence penalty from the Critical Insights section.

```python
r = np.random.default_rng(0)
X0 = r.normal([-1, 1], 0.6, size=(50, 2)); X1 = r.normal([1, -1], 0.6, size=(50, 2))
X = np.r_[X0, X1]; y = np.r_[np.zeros(50, int), np.ones(50, int)]
Xo = r.normal([7, -6], 0.5, size=(20, 2))
Xa = np.r_[X, Xo]; ya = np.r_[y, np.ones(20, int)]
acc_before = np.mean(predict_ls(fit_ls(X, y, 2), X) == y)
acc_after = np.mean(predict_ls(fit_ls(Xa, ya, 2), X) == y)
acc_logreg_after = LogisticRegression().fit(Xa, ya).score(X, y)
print(acc_before, acc_after, acc_logreg_after)
```

</details>

### Exercise 17 · Perceptron from scratch
*💻 Code · ★★☆*

Implement the Perceptron (labels $t_n\in\{-1,+1\}$, augmented $\tilde{\mathbf{x}}_n$): cycle through the data and whenever
$t_n\,\tilde{\mathbf{w}}^T\tilde{\mathbf{x}}_n \le 0$ update $\tilde{\mathbf{w}} \leftarrow \tilde{\mathbf{w}} + t_n\tilde{\mathbf{x}}_n$. Stop after an epoch without mistakes
(or after `max_epochs`). Train on the separable data `X_sep, t_sep` below and store the number of training errors and of epochs used.
Run it once more on the data with the outliers of the previous exercise: does it still find a separating line?

In [ ]:
r = np.random.default_rng(3)
X_sep = np.r_[r.normal([-2, 0], 0.7, size=(40, 2)), r.normal([2, 1], 0.7, size=(40, 2))]
t_sep = np.r_[-np.ones(40), np.ones(40)]

def perceptron(X, t, max_epochs=100):
    # TODO: return (w_tilde, n_epochs)
    return None, None

w_pc, epochs_pc = perceptron(X_sep, t_sep)
n_errors_pc = None

check('zero training errors (separable data)', n_errors_pc, int(np.sum(Perceptron(max_iter=1000, tol=None).fit(X_sep, t_sep).predict(X_sep) != t_sep)))

<details>
<summary><b>💡 Hint</b></summary>

`Xt = np.c_[np.ones(len(X)), X]`; loop `for n in range(N)`, count mistakes per epoch.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The perceptron convergence theorem guarantees a separating hyperplane after finitely many mistakes when the data is linearly separable (sklearn's
`Perceptron` reaches 0 errors too, which is what the check uses). It only reacts to *misclassified* points, so far-away correct points (Exercise 16) do not
move it at all. On non-separable data it never stops cycling; that is why `max_epochs` is needed.

```python
r = np.random.default_rng(3)
X_sep = np.r_[r.normal([-2, 0], 0.7, size=(40, 2)), r.normal([2, 1], 0.7, size=(40, 2))]
t_sep = np.r_[-np.ones(40), np.ones(40)]

def perceptron(X, t, max_epochs=100):
    Xt = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xt.shape[1])
    for epoch in range(1, max_epochs + 1):
        mistakes = 0
        for n in range(len(t)):
            if t[n] * (w @ Xt[n]) <= 0:
                w += t[n] * Xt[n]
                mistakes += 1
        if mistakes == 0:
            return w, epoch
    return w, max_epochs

w_pc, epochs_pc = perceptron(X_sep, t_sep)
n_errors_pc = int(np.sum(np.sign(np.c_[np.ones(len(X_sep)), X_sep] @ w_pc) != t_sep))
print('epochs:', epochs_pc, 'errors:', n_errors_pc)
```

</details>

### Exercise 18 · Self Study 1, Exercise 4: who gets the outlier right?
*💻 Code · ★★★*

Recreate the synthetic set of Self Study 1: 100 points from $\mathcal N((-2,-2), I)$ (class 0), 100 from $\mathcal N((3.5, 0), 0.1I)$ (class 1), both with
`random_state=1`, plus one extra class-1 point at $(0.5, -1)$. The data is linearly separable.

Find (a) a classifier that predicts the outlier as class **0** and (b) one that predicts it as class **1** while classifying *all* points correctly.
Store the predictions for the outlier in `pred_outlier_a`, `pred_outlier_b` and the training accuracy of (b) in `acc_b`.
Which of your classifiers would you trust more on new data, and why?

In [ ]:
g1 = mvn(mean=(-2, -2), cov=((1, 0), (0, 1))).rvs(random_state=1, size=100)
g2 = mvn(mean=(3.5, 0), cov=((0.1, 0), (0, 0.1))).rvs(random_state=1, size=100)
X_ss = np.vstack([g1, g2, (0.5, -1)])
Y_ss = np.hstack([np.zeros(100), np.ones(101)]).astype(int)

pred_outlier_a = None
pred_outlier_b = None
acc_b = None

check('(a) outlier -> class 0', pred_outlier_a, 0)
check('(b) outlier -> class 1', pred_outlier_b, 1)
check('(b) separates everything', acc_b, 1.0)

<details>
<summary><b>💡 Hint 1</b></summary>

Your least-squares classifier (or `LogisticRegression()` with default regularisation) is a candidate for (a).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For (b) you need a classifier that insists on separating the data: a (nearly) hard-margin `SVC(kernel='linear', C=1e4)` or the perceptron.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Least squares, LDA, default logistic regression and a soft-margin SVM with $C=1$ all sacrifice the single outlier (accuracy $200/201$); a hard-margin
linear SVM (large $C$) or the perceptron separate all 201 points. "For a good reason": the outlier is one point deep inside class-0 territory and is
quite possibly label noise, so the boundary that ignores it is arguably better on new data. Which behaviour you want is a modelling decision
(the soft-margin $C$ of [[Lecture Notes 1-12#Lecture 3: Kernel Support Vector Machines]] makes it explicit).

```python
g1 = mvn(mean=(-2, -2), cov=((1, 0), (0, 1))).rvs(random_state=1, size=100)
g2 = mvn(mean=(3.5, 0), cov=((0.1, 0), (0, 0.1))).rvs(random_state=1, size=100)
X_ss = np.vstack([g1, g2, (0.5, -1)])
Y_ss = np.hstack([np.zeros(100), np.ones(101)]).astype(int)

W_ss = fit_ls(X_ss, Y_ss, 2)
pred_outlier_a = int(predict_ls(W_ss, X_ss[-1:])[0])
svc = SVC(kernel='linear', C=1e4).fit(X_ss, Y_ss)
pred_outlier_b = int(svc.predict(X_ss[-1:])[0])
acc_b = svc.score(X_ss, Y_ss)
print('LS:', pred_outlier_a, ' hard-margin SVM:', pred_outlier_b, acc_b)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*